# 📰 News Category Classification Project

This notebook implements a complete Machine Learning pipeline to classify HuffPost news articles into 19 categories based on their **headlines** and **short descriptions**. 

**Models Evaluated**:
- Logistic Regression
- Multinomial Naive Bayes
- Random Forest
- Linear Support Vector Machine (SVM)


In [ ]:
import os
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

import nltk

warnings.filterwarnings("ignore")


### Download NLTK Dependencies
We need stopwords and the WordNet lemmatizer corpus.

In [ ]:
def _ensure_nltk_data():
    packages = ["stopwords", "wordnet"]
    for pkg in packages:
        try:
            nltk.data.find(f"corpora/{pkg}")
        except Exception:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass

_ensure_nltk_data()

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


### 1. Data Loading & Exploration
Load the dataset from the CSV file and look at the category distribution.

In [ ]:
dataset_file = "news_category_dataset.csv"
if not os.path.exists(dataset_file):
    dataset_file = "news_catagery_dataset.csv"

df = pd.read_csv(dataset_file)
print(f"Dataset Shape: {df.shape}")
print(f"Categories:\n{df['category'].value_counts()}")


In [ ]:
plt.figure(figsize=(14, 6))
counts = df["category"].value_counts()
sns.barplot(x=counts.index, y=counts.values, palette="viridis")
plt.title("News Category Distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 2. Data Preprocessing & Text Cleaning
We fill missing authors, merge headlines with short descriptions, and apply standard NLP cleaning (lowercase, remove punctuation, remove stopwords, and lemmatize).

In [ ]:
# Preprocessing
df["authors"] = df["authors"].fillna("Unknown")
df["text"] = df["headline"].astype(str) + " " + df["short_description"].astype(str)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words and len(w) > 2]
    return " ".join(words)

print("Applying text cleaning...")
df["text_clean"] = df["text"].apply(clean_text)
df = df[df["text_clean"].str.len() > 0]
print("Cleaning complete. Data shape:", df.shape)


### 3. TF-IDF Vectorization and Train/Test Split

In [ ]:
X = df["text_clean"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Train matrix: {X_train_tfidf.shape}")
print(f"Test matrix: {X_test_tfidf.shape}")


### 4. Model Training and Evaluation
Train 4 different models using balanced class weights.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Multinomial NB": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "Linear SVM": LinearSVC(max_iter=2000, class_weight="balanced", random_state=42),
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    
    results[name] = {"model": model, "accuracy": acc, "f1_score": f1, "y_pred": y_pred}
    print(f"  Accuracy: {acc:.4f} | F1 Score: {f1:.4f}\n")


### 5. Final Results & Best Model

In [ ]:
best_name = max(results, key=lambda k: results[k]["f1_score"])
print(f"🏆 Best Model: {best_name} (F1: {results[best_name]['f1_score']:.4f})")

# Save model
best_model = results[best_name]["model"]
joblib.dump(best_model, "best_model.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
print("Saved best model to disk.")
